# 01 — Bronze Ingestion

Reads F1 data from Parquet files uploaded to the Databricks workspace
(see `scripts/fetch_and_upload.py`) and MERGEs each table into its Bronze
Delta table with CDF and Liquid Clustering enabled.

**Sources**: Jolpica-F1 API (race results, standings, qualifying, pit stops)
and OpenF1 API (lap timing, tyre stints).

**Write mode**: MERGE on natural key — re-running this notebook for the same
season/round is fully idempotent. Post-race steward penalties that change
a driver's final position or points are captured as updates on the next run.

In [ ]:
%pip install /Workspace/Users/pravbala30@protonmail.com/.bundle/f1-intelligence/dev/artifacts/.internal/f1_intelligence-0.1.0-py3-none-any.whl --quiet
dbutils.library.restartPython()

In [ ]:
# Parameters
dbutils.widgets.text("catalog", "f1_intelligence")
dbutils.widgets.text("schema",  "f1_dev")
dbutils.widgets.text("season",  "2024")
dbutils.widgets.text("mode",    "full")  # full | incremental

CATALOG = dbutils.widgets.get("catalog")
SCHEMA  = dbutils.widgets.get("schema")
SEASON  = dbutils.widgets.get("season")
MODE    = dbutils.widgets.get("mode")

print(f"catalog={CATALOG}  schema={SCHEMA}  season={SEASON}  mode={MODE}")

In [ ]:
# Setup
from utils.helpers import (
    get_or_create_catalog_schema,
    merge_delta,
    add_metadata_columns,
    print_table_stats,
)
from utils.schema import merge_keys
from utils.validators import validate_bronze_f1

get_or_create_catalog_schema(spark, CATALOG, SCHEMA)

WORKSPACE_DATA = "/Workspace/Users/pravbala30@protonmail.com/.bundle/f1-intelligence/dev/f1_data"

def full_name(table):
    return f"{CATALOG}.{SCHEMA}.{table}"

def ingest(parquet_name, table_name, source):
    """
    Read a Parquet file from the workspace, attach metadata, validate, and MERGE into a Bronze Delta table.

    Steps:
        1. Read the pre-fetched Parquet from WORKSPACE_DATA/<parquet_name>.parquet.
        2. Append ingested_at and source_name columns via add_metadata_columns().
        3. Run the soft-fail Bronze validator — logs warnings on null MERGE keys.
        4. MERGE into the fully-qualified Delta table (creates it on first run).
        5. Print row count and Delta version for observability.

    Example:
        ingest("bronze_race_results", "bronze_race_results", "jolpica_f1")
        # Reads  : /Workspace/.../f1_data/bronze_race_results.parquet
        # Writes : f1_intelligence.f1_dev.bronze_race_results  (MERGE)
        # Prints : f1_intelligence.f1_dev.bronze_race_results: 480 rows, Delta version 1
    """
    path = f"{WORKSPACE_DATA}/{parquet_name}.parquet"
    df_raw = spark.read.parquet(path)
    df = add_metadata_columns(df_raw, source)
    result = validate_bronze_f1(df, table_name)
    result.log(table_name)
    merge_delta(spark, df, full_name(table_name), merge_keys[table_name])
    print_table_stats(spark, full_name(table_name))

print("Setup complete")

In [ ]:
# Ingest Jolpica tables
print("=== Jolpica-F1 ingestion ===")

ingest("bronze_race_schedule",         "bronze_race_schedule",         "jolpica_f1")
ingest("bronze_race_results",          "bronze_race_results",          "jolpica_f1")
ingest("bronze_qualifying",            "bronze_qualifying",            "jolpica_f1")
ingest("bronze_pit_stops",             "bronze_pit_stops",             "jolpica_f1")
ingest("bronze_driver_standings",      "bronze_driver_standings",      "jolpica_f1")
ingest("bronze_constructor_standings", "bronze_constructor_standings", "jolpica_f1")

In [ ]:
# Ingest OpenF1 tables
print("=== OpenF1 ingestion ===")

ingest("bronze_laps",   "bronze_laps",   "openf1")
ingest("bronze_stints", "bronze_stints", "openf1")

In [ ]:
# Summary
print("\n=== Bronze ingestion complete ===")
tables = [
    "bronze_race_schedule", "bronze_race_results", "bronze_qualifying",
    "bronze_pit_stops", "bronze_driver_standings", "bronze_constructor_standings",
    "bronze_laps", "bronze_stints",
]
for t in tables:
    print_table_stats(spark, full_name(t))

dbutils.notebook.exit("bronze_ingestion_complete")